# Experiments: extending the Saito criterion to $\ell+2$ generators

The chapters below explore (a) the full $(\ell+2)$-tuple cofactor structure, (b) a conjectural height bound $\operatorname{ht}\,I_\mathcal{A}(G)\ge\operatorname{pd}_S D(\mathcal{A})+2$, and (c) a unified Plücker-tensor formulation of the Saito-type criterion that interpolates between the classical, SPOG, and $(\ell+2)$ cases.


In [14]:
from IPython.display import Math, display
import sage.misc.latex as sage_latex
import itertools
import sys
from sage.modules.free_module_element import FreeModuleElement_generic_dense as module_elem
from sage.libs.singular.function_factory import singular_function
syz = singular_function("syz")

from pathlib import Path

def _find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'src' / 'hyperplane_arrangements').is_dir():
            return path
    raise RuntimeError('Run this notebook from inside the repository or install the package with sage -pip install -e .')

ROOT = _find_repo_root()
src = ROOT / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
from hyperplane_arrangements import *
# Re-import utility functions not re-exported by the package __init__.
from hyperplane_arrangements.utils import (
    coordinate_vectors, create_generic_arrangement, sk_expo, exponent_to_polynomial,
)

try:
    original_show = show
except NameError:
    raise SystemExit("Switch to the SageMath kernel and re-run.")

def vscode_show(*args, **kwargs):
    for expr in args:
        try:
            if hasattr(expr, '_latex_'):
                display(Math(expr._latex_()))
            else:
                display(Math(sage_latex.latex(expr)))
        except Exception as e:
            print(f"LaTeX rendering failed: {e}")
            original_show(expr, **kwargs)
show = vscode_show
print(sage.version.banner)

def _generator_list(A, generators=None):
    return list(A.minimal_generators() if generators is None else generators)


def report_scaled_minor_checks(A, generators=None, label="G", verify=False):
    """Print the full scaled-minor height and check all ell+1 sublists."""
    G = _generator_list(A, generators)
    ell = A.n
    p = len(G)
    I = A.scaled_minor_ideal(G)
    ht = A.scaled_minor_ideal_height(G)
    print(f"{label}: p={p}, ell={ell}, ht I_A({label})={ht}, proper={not I.is_one()}")

    if p < ell + 1:
        print("  no ell+1 sublists to test")
        return []

    subset_results = []
    for J in itertools.combinations(range(p), ell + 1):
        sub = [G[j] for j in J]
        res = A.check_generalized_saito(generators=sub, verify=verify, verbose=False)
        subset_results.append((J, res))

    height_counts = {}
    for _, res in subset_results:
        height_counts[res['height']] = height_counts.get(res['height'], 0) + 1
    n_criterion = sum(1 for _, res in subset_results if res['criterion_applies'])
    n_spog = sum(1 for _, res in subset_results if res['predicts_minimal_spog'])
    print(f"  checked C({p},{ell + 1}) = {len(subset_results)} sublists")
    print(f"  height distribution: {height_counts}")
    print(f"  criterion applies: {n_criterion}; predicts minimal SPOG: {n_spog}")
    if verify:
        n_generate = sum(1 for _, res in subset_results if res.get('actually_generates'))
        print(f"  actually generates: {n_generate}")
    if len(subset_results) <= 20:
        for J, res in subset_results:
            print(f"    {J}: ht={res['height']}, criterion={res['criterion_applies']}, predicts_min_SPOG={res['predicts_minimal_spog']}")
    return subset_results


SageMath version 10.7, Release Date: 2025-08-09


| Section (Link) | What it is meant to show |
|---|---|
| [Extended Investigation: the Full $(\ell+2)$-Tuple](#Extended-Investigation:-the-Full-$(\ell+2)$-Tuple) | The SPOG-style criterion never fires on $(\ell+1)$-subsets when $|G_{\min}|=\ell+2$; we set up the full-$(\ell+2)$ cofactor / relation structure. |
| [Detailed View of the Example-10 Arrangement](#Detailed-View-of-the-Example-10-Arrangement) | Explicit relation matrix $N$, every $g_I$, and the Plücker identity verified directly. |
| [A Catalogue of PD=1, $\ell+2$ Arrangements](#A-Catalogue-of-PD=1,-$\ell+2$-Arrangements) | A handful of $\operatorname{pd}=1$, $\ell+2$ examples in dim 3 and 4 collected for cross-comparison. |
| [Tentative Conjecture (extended Saito-type criterion)](#Tentative-Conjecture-(extended-Saito-type-criterion)) | A first conjecture using height + Plücker-style hypothesis, with stress tests by row perturbation. |
| [Example 12: Unified Saito-type Criterion via the Plücker Tensor](#Example-12:-Unified-Saito-type-Criterion-via-the-Plücker-Tensor) | $g_I$-only formulation: $\widetilde{g}\in\bigwedge^{p-\ell} S^p$ decomposability.  Necessary at every $k$; sufficient for $k\le 1$ (recovers Saito + Thm 1.1); a counterexample shows it is **not** sufficient for $k\ge 2$. |


### Extended Investigation: the Full $(\ell+2)$-Tuple

The natural question is: what does the criterion look like when applied to the **full** $(\ell+2)$-tuple?

Let $M$ be the $(\ell+2)\times\ell$ derivation matrix and $N$ a $2\times(\ell+2)$ relation matrix
($NM=0$).  We now have

* $\binom{\ell+2}{2}$ scaled cofactors $g_I = \det(M_I)/Q$ indexed by $\ell$-subsets $I\subset[\ell+2]$,
* $\binom{\ell+2}{2}$ maximal minors $n_T = \det(N_T)$ of $N$ indexed by $2$-subsets $T\subset[\ell+2]$.

Bijection by complement: $I\leftrightarrow T=[\ell+2]\setminus I$.

**Degree check.** With $D_1, D_2$ the relation degrees and $d_i$ the generator degrees, the Hilbert
series of $D(\mathcal{A})$ forces $\sum d_i - D_1 - D_2 = |\mathcal{A}|$.  Combining this with
$\deg g_I = \sum_{i\in I}d_i - |\mathcal{A}|$ and $\deg n_{[p]\setminus I} = D_1+D_2 - \sum_{i\in I}d_i$
gives $\deg g_I = \deg n_{[\ell+2]\setminus I}$.  So a Pluecker-style identity
$g_I = c\cdot\varepsilon(I)\cdot n_{[\ell+2]\setminus I}$ with a constant $c\in\mathbb{K}^*$ is at least
*numerically* consistent.  We verify it directly.


In [16]:
# Helpers for the PD=1, ell+2 experiment.
def pd_DA(A):
    """Projective dimension of D(A) from the minimal free resolution."""
    return A.free_resolution()._length - 1

def relations_matrix(A, generators=None):
    """Compute the matrix N of syzygies for the candidate generators (k x p)."""
    if generators is None:
        gens = list(A.minimal_generators())
    else:
        gens = list(generators)
    M = matrix([A._to_vector(g) for g in gens])
    p = M.nrows()
    rels = syz(Sequence([module_elem(A.S**A.n, tuple(M[i])) for i in range(p)]))
    if not rels:
        return matrix(A.S, 0, p)
    return matrix(A.S, [list(r) for r in rels])

def plucker_test(A, generators=None):
    """Test whether g_I and det(N_{[p]\\I}) are scalar-proportional (Pluecker identity).
    Returns ('hold', g0//d, n0//d), ('fail', I, T), 'degenerate', or None."""
    if generators is None:
        G = list(A.minimal_generators())
    else:
        G = list(generators)
    p = len(G); ell = A.n
    M = matrix([A._to_vector(g) for g in G])
    N = relations_matrix(A, G)
    k = N.nrows()
    if k != p - ell:
        return None
    check_saito_criterion = A.saito_coefficients(generators=G, as_dict=True, signed=True)
    n_minors = {T: N[:, list(T)].det() for T in itertools.combinations(range(p), k)}
    pairs = []
    for I_sub, g_I in check_saito_criterion.items():
        T = tuple(j for j in range(p) if j not in I_sub)
        pairs.append((I_sub, T, g_I, n_minors[T]))
    baseline = next(((I, T, g, n) for I, T, g, n in pairs if g != 0 and n != 0), None)
    if baseline is None:
        return ("degenerate", None, None)
    I0, T0, g0, n0 = baseline
    for I_sub, T, g_I, nT in pairs:
        lhs, rhs = g_I * n0, g0 * nT
        if lhs != rhs and lhs != -rhs:
            return ("fail", I_sub, T)
    d = gcd(g0, n0)
    return ("hold", g0 // d, n0 // d)

def report_pd1_lplus2(A, generators=None, label=""):
    """Diagnostics report for one arrangement in the PD=1, ell+2 setting."""
    if generators is None:
        G = list(A.minimal_generators())
    else:
        G = list(generators)
    p = len(G); ell = A.n
    M = matrix([A._to_vector(g) for g in G])
    N = relations_matrix(A, G)
    k = N.nrows()
    pd = pd_DA(A)
    ht = A.scaled_minor_ideal_height(G)
    pd_bound = pd + 2
    bound_ok = (ht == float('inf')) or (ht >= pd_bound)
    rel_degs = sorted(max((c.degree() for c in N.row(i) if c != 0), default=-1) for i in range(k))

    print(f"  {label}")
    print(f"    ell={ell}, |A|={A.num_planes}, |G|={p}, degrees={A.degrees()}")
    print(f"    pd D(A)={pd}, #relations={k}, relation degrees={rel_degs}")
    print(f"    ht I_A(G)={ht}  (Ex.11 bound: pd+2={pd_bound}) -> {bound_ok}")
    print(f"    N*M==0: {(N * M).is_zero()}")
    if k == p - ell and k >= 1:
        res = plucker_test(A, G)
        if res is None:
            print(f"    Pluecker test skipped.")
        elif res[0] == "hold":
            _, gr, nr = res
            print(f"    Pluecker identity HOLDS: g_I : det(N_T) = ({gr}) : ({nr})")
        elif res[0] == "degenerate":
            print(f"    Pluecker test: no pair (g_I, n_T) both nonzero (degenerate).")
        else:
            print(f"    Pluecker identity FAILS at I={res[1]}, T={res[2]}")

### Detailed View of the Example-10 Arrangement

We dump the relation matrix $N$, every $g_I$, and verify the Pluecker identity
$g_I = c\cdot\varepsilon(I)\cdot \det(N_{[\ell+2]\setminus I})$ explicitly.


In [18]:
A_pd1 = HyperplaneArrangement([[1,-2,0],[2,1,1],[0,1,0],[2,-1,2],[-1,0,-1],[-2,2,0],[2,2,-1]])
report_pd1_lplus2(A_pd1, label="A_pd1 (Example 10 arrangement)")
print()

N = relations_matrix(A_pd1)
print("Relation matrix N (2 x 5):")
show(N)

print("\nAll scaled cofactors g_I and complementary 2-minors det(N_T):")
check_saito_criterion = A_pd1.saito_coefficients(as_dict=True, signed=True)
for I_sub in sorted(check_saito_criterion.keys()):
    T = tuple(j for j in range(5) if j not in I_sub)
    g = check_saito_criterion[I_sub]
    nT = N[:, list(T)].det()
    dg = g.degree() if g != 0 else -1
    dn = nT.degree() if nT != 0 else -1
    print(f"  I={I_sub} <-> T={T}:  deg g_I={dg}, deg det(N_T)={dn}")
    print(f"     g_I       = {g}")
    print(f"     det(N_T)  = {nT}")
    print(F"     g_I/n_T = {g/nT if nT != 0 else 'undefined'}")

  A_pd1 (Example 10 arrangement)
    ell=3, |A|=7, |G|=5, degs=[1, 4, 4, 5, 5]
    pd D(A)=1, #relations=2, relation degs=[5, 5]
    ht I_A(G)=3  (Ex.11 bound: pd+2=3) -> True
    N*M==0: True
    Pluecker identity HOLDS: g_I : det(N_T) = (-49) : (-65)

Relation matrix N (2 x 5):


<IPython.core.display.Math object>


All scaled cofactors g_I and complementary 2-minors det(N_T):
  I=(0, 1, 2) <-> T=(3, 4):  deg g_I=2, deg det(N_T)=2
     g_I       = -196*x0^2 - 294*x0*x1 - 98*x1^2 - 49*x1*x2 + 49*x2^2
     det(N_T)  = -260*x0^2 - 390*x0*x1 - 130*x1^2 - 65*x1*x2 + 65*x2^2
     g_I/n_T = 49/65
  I=(0, 1, 3) <-> T=(2, 4):  deg g_I=3, deg det(N_T)=3
     g_I       = -18326*x1^3 - 19110*x0*x1*x2 + 21707*x1^2*x2 - 7644*x0*x2^2 - 4753*x1*x2^2 + 490*x2^3
     det(N_T)  = -24310*x1^3 - 25350*x0*x1*x2 + 28795*x1^2*x2 - 10140*x0*x2^2 - 6305*x1*x2^2 + 650*x2^3
     g_I/n_T = 49/65
  I=(0, 1, 4) <-> T=(2, 3):  deg g_I=3, deg det(N_T)=3
     g_I       = -1274*x0*x1^2 + 5194*x1^3 + 6370*x0*x1*x2 - 7399*x1^2*x2 + 2548*x0*x2^2 + 1715*x1*x2^2 - 98*x2^3
     det(N_T)  = -1690*x0*x1^2 + 6890*x1^3 + 8450*x0*x1*x2 - 9815*x1^2*x2 + 3380*x0*x2^2 + 2275*x1*x2^2 - 130*x2^3
     g_I/n_T = 49/65
  I=(0, 2, 3) <-> T=(1, 4):  deg g_I=3, deg det(N_T)=3
     g_I       = 7644*x0*x1^2 + 42630*x1^3 + 23422*x0*x1*x2 - 54537*x1^2*x2 +

### A Catalogue of PD=1, $\ell+2$ Arrangements

For a meaningful conjecture we need multiple examples.  Below we collect a few PD=1, $\ell+2$
arrangements (dimension 3 and dimension 4) and run the same report on each, so that the height
bound, the Pluecker identity, and the relation structure can be compared across cases.


In [19]:
# Direct examples (dim 3) and a search over deletions for more cases.
catalogue = []

catalogue.append(("dim 3 (Example 10 arrangement)",
                  HyperplaneArrangement([[1,-2,0],[2,1,1],[0,1,0],
                                 [2,-1,2],[-1,0,-1],[-2,2,0],[2,2,-1]])))

# Hand-constructed dim-3 candidates: we only keep those with pd=1 and |G|=ell+2.
for tag, mat in [
    ("variant A", [[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,-1,2],[1,2,-1],[2,1,1]]),
    ("variant B", [[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2],[1,2,1],[3,1,1]]),
    ("variant C", [[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2],[1,3,1],[2,1,3]]),
]:
    A_try = HyperplaneArrangement(mat)
    if pd_DA(A_try) == 1 and len(A_try.degrees()) == A_try.n + 2:
        catalogue.append((f"dim 3 ({tag})", A_try))

# Search by deletion from larger dim-3 arrangements.
for mat in [
    coordinate_vectors(3) + [[1,1,1],[1,1,2],[1,2,1],[2,1,1],[1,2,3],[1,3,2]],
    coordinate_vectors(3) + [[1,1,1],[1,-1,1],[1,1,-1],[1,2,1],[2,1,1],[1,1,2]],
]:
    big = HyperplaneArrangement(mat)
    for i in range(big.num_planes):
        B = big.deletion([i])
        if pd_DA(B) == 1 and len(B.degrees()) == B.n + 2:
            catalogue.append((f"dim 3 (deletion of plane {i} from |A|={big.num_planes})", B))

# Try a dim-4 case by deletion.
big4 = HyperplaneArrangement(coordinate_vectors(4) + [[1,1,1,0],[1,1,0,1],[1,0,1,1],[1,1,1,1],[1,-1,0,0]])
for i in range(big4.num_planes):
    B = big4.deletion([i])
    if pd_DA(B) == 1 and len(B.degrees()) == B.n + 2:
        catalogue.append((f"dim 4 (deletion of plane {i} from |A|={big4.num_planes})", B))
        break

# Deduplicate by (n, |A|, degrees).
seen = set(); unique = []
for label, B in catalogue:
    key = (B.n, B.num_planes, tuple(B.degrees()))
    if key not in seen:
        seen.add(key); unique.append((label, B))

print(f"Distinct PD=1, ell+2 examples to inspect: {len(unique)}\n")
for label, B in unique:
    report_pd1_lplus2(B, label=label)
    print()

Distinct PD=1, ell+2 examples to inspect: 5

  dim 3 (Example 10 arrangement)
    ell=3, |A|=7, |G|=5, degs=[1, 4, 4, 5, 5]
    pd D(A)=1, #relations=2, relation degs=[5, 5]
    ht I_A(G)=3  (Ex.11 bound: pd+2=3) -> True
    N*M==0: True
    Pluecker identity HOLDS: g_I : det(N_T) = (-49) : (-65)

  dim 3 (deletion of plane 1 from |A|=9)
    ell=3, |A|=8, |G|=5, degs=[1, 5, 5, 5, 6]
    pd D(A)=1, #relations=2, relation degs=[2, 2]
    ht I_A(G)=3  (Ex.11 bound: pd+2=3) -> True
    N*M==0: True
    Pluecker identity HOLDS: g_I : det(N_T) = (-1024) : (-2560)

  dim 3 (deletion of plane 7 from |A|=9)
    ell=3, |A|=8, |G|=5, degs=[1, 5, 5, 5, 5]
    pd D(A)=1, #relations=2, relation degs=[1, 2]
    ht I_A(G)=3  (Ex.11 bound: pd+2=3) -> True
    N*M==0: True
    Pluecker identity HOLDS: g_I : det(N_T) = (192) : (48)

  dim 3 (deletion of plane 1 from |A|=9)
    ell=3, |A|=8, |G|=5, degs=[1, 4, 5, 5, 5]
    pd D(A)=1, #relations=2, relation degs=[2, 2]
    ht I_A(G)=3  (Ex.11 bound: pd+2=3

### Tentative Conjecture (extended Saito-type criterion)

**Conjecture (extended Saito-type criterion, PD=1 with $\ell+2$ generators).**
Let $\mathcal{A}$ be a central arrangement in $\mathbb{K}^\ell$ with $\operatorname{pd}_S D(\mathcal{A})=1$.
Let $G=\{\theta_1,\dots,\theta_{\ell+2}\}\subset D(\mathcal{A})$ be homogeneous derivations with derivation
matrix $M$, and pick a $2\times(\ell+2)$ matrix $N$ over $S$ with $NM=0$ whose rows span the kernel of
the transpose of $M$ rationally.  If both

(i)  $\operatorname{ht}\,I_\mathcal{A}(G)\ge 3$ (equivalently $\operatorname{ht}\,\langle g_I\rangle\ge 3$), **and**

(ii)  $\operatorname{ht}\,\langle \det N_T : |T|=2 \rangle \ge 2$,

then $G$ generates $D(\mathcal{A})$ and its syzygy module is freely generated by the two rows of $N$.

Hypothesis (i) is the Example-11 inequality at $\operatorname{pd}=1$.  Hypothesis (ii) — codimension at least
$2$ for the ideal of $2\times2$ minors of $N$ — is the *codim-2* condition that, via Hilbert--Burch /
Buchsbaum--Eisenbud, makes the determinantal complex exact and identifies the row-space of $N$ as the
syzygies of $M$.  Together they form the natural successor of "$\operatorname{ht}\ge3$ in the SPOG case"
that already incorporates the Pluecker identity established empirically above.

The stress test below shows that the height-only hypothesis (i) is **not** sufficient on its own: a
candidate set $G$ with $\operatorname{ht}\,I_\mathcal{A}(G)\ge 3$ but a degenerate $N$ (rank below $2$, or
ideal of $2$-minors with low height) can fail to generate.  The role of hypothesis (ii) is to forbid
exactly that degeneracy.


In [ ]:
# Stress-test the conjecture by perturbing the minimal generators and checking
# the (height, Pluecker-height) hypothesis vs actual generation.
def aux_height(A, N):
    """ht of the ideal of maximal (k x k) minors of N (k = N.nrows())."""
    p = N.ncols(); k = N.nrows()
    if k == 0 or p < k:
        return float('inf')
    minors = [N[:, list(T)].det() for T in itertools.combinations(range(p), k)]
    minors = [m for m in minors if m != 0]
    if not minors:
        return 0
    J = A.S.ideal(minors)
    if J.is_one():
        return float('inf')
    return A.n - J.dimension()

A = HyperplaneArrangement([[1,-2,0],[2,1,1],[0,1,0],[2,-1,2],[-1,0,-1],[-2,2,0],[2,2,-1]])
G0 = list(A.minimal_generators().gens)
x = A.v

cases = [
    ("baseline minimal G",                              lambda Gp: Gp),
    ("G[1] += x0*G[0]",                                 lambda Gp: [Gp[0], Gp[1] + x[0]*Gp[0], Gp[2], Gp[3], Gp[4]]),
    ("G[3] += x1*G[2]",                                 lambda Gp: [Gp[0], Gp[1], Gp[2], Gp[3] + x[1]*Gp[2], Gp[4]]),
    ("G[4] += (x0+x2)*G[1]",                            lambda Gp: [Gp[0], Gp[1], Gp[2], Gp[3], Gp[4] + (x[0]+x[2])*Gp[1]]),
    ("G[4] := G[3]  (degenerate: repeat)",              lambda Gp: [Gp[0], Gp[1], Gp[2], Gp[3], Gp[3]]),
    ("G[4] := x0*G[3]  (degenerate: in submodule)",     lambda Gp: [Gp[0], Gp[1], Gp[2], Gp[3], x[0]*Gp[3]]),
    ("drop G[4], replace with x0*G[1]",                 lambda Gp: [Gp[0], Gp[1], Gp[2], Gp[3], x[0]*Gp[1]]),
]

print(f"{'case':50s} ht I_A(G)  rk N  ht <det N_T>  generates  (i)&(ii)?")
print("-" * 105)
for tag, modifier in cases:
    G = modifier(list(G0))
    ht_I = A.scaled_minor_ideal_height(G)
    N = relations_matrix(A, G)
    k = N.nrows()
    ht_N = aux_height(A, N) if k >= 1 else float('inf')
    actually = A.candidate_generates(G)
    cond_i  = (ht_I == float('inf')) or (isinstance(ht_I, (int, Integer)) and ht_I >= 3)
    cond_ii = (k == 2) and isinstance(ht_N, (int, Integer)) and ht_N >= 2
    both    = cond_i and cond_ii
    note = ""
    if both and not actually:
        note = "  <-- COUNTEREXAMPLE to conjecture!"
    elif both and actually:
        note = "  predicts generation: OK"
    elif not both and actually:
        note = "  generates but criterion silent"
    print(f"  {tag:48s} {str(ht_I):9s}  {k:3d}   {str(ht_N):11s}  {str(actually):9s}  {both}{note}")

## Example 12: Unified Saito-type Criterion via the Plücker Tensor

The classical Saito criterion ($p=\ell$), Theorem 1.1 ($p=\ell+1$), and the Example-10 extension ($p=\ell+2$) all admit a single $g_I$-only formulation.  Bundle the scaled minors into the alternating tensor

$$
\widetilde{g} \;=\; \sum_{|T|=k}\,(-1)^{\sigma(T)}\,g_{[p]\setminus T}\,e_{t_1}\wedge\cdots\wedge e_{t_k} \;\in\; \bigwedge^{k} S^{p}, \qquad k=p-\ell,
$$
with the standard subset sign $\sigma(T)=\sum_j(t_j-j)$ used by `saito_coefficients(signed=True)`.

This is implemented in the package as `A.scaled_minor_tensor(G)` (returns `{T: \tilde g_T}`).  For $k=2$ it is encoded as the antisymmetric $p\times p$ matrix $A.scaled\_minor\_matrix(G)$ with entries $\widetilde{g}_{a,b}=(-1)^{a+b+1}\,g_{[p]\setminus\{a,b\}}$.

**Cofactor identity.**  $\widetilde{g}\cdot M=0$ holds *unconditionally* — it is a Laplace expansion identity.  Equivalently, contracting $\widetilde{g}$ with any $k-1$ rows of $M$ gives a syzygy of $G$.  The rows of the antisymmetric matrix view ($k=2$) are therefore syzygies of $G$ for free.

The $g_I$-only invariants relevant for the unified statement are then:

* **(i)** $\operatorname{ht}\,I_\mathcal{A}(G)\ge 3$ — the codimension condition on the scaled-minor ideal.
* **(ii)** $\widetilde{g}$ is **decomposable** in $\bigwedge^k S^p$, i.e., $\widetilde{g}=n_1\wedge\cdots\wedge n_k$ for some $n_i\in S^p$, equivalently the Grassmann–Plücker quadrics on $\{g_I\}$ vanish.

Concretely for $k=2$ the Plücker quadrics are
$$
g_{[p]\setminus\{a,b\}}\,g_{[p]\setminus\{c,d\}} - g_{[p]\setminus\{a,c\}}\,g_{[p]\setminus\{b,d\}} + g_{[p]\setminus\{a,d\}}\,g_{[p]\setminus\{b,c\}} = 0
$$
for every $4$-subset $\{a,b,c,d\}\subset[p]$ — equivalently, all $4\times 4$ Pfaffians of the antisymmetric matrix $\widetilde{g}_{a,b}$ vanish, equivalently $\widetilde{g}$ has rank $\le 2$ over $\operatorname{Frac}(S)$.

**Both (i) and (ii) are *necessary* for $G$ to generate $D(\mathcal{A})$.**  Indeed, suppose $G$ generates $D(\mathcal{A})$.  Then $M$ has rank $\ell$ over $\operatorname{Frac}(S)$, the syzygy module is rank $k$, and the rows of $\widetilde{g}$ are syzygies (by the cofactor identity).  Any $k$ rationally-independent rows give a decomposition of $\widetilde{g}$ as an alternating $k$-tensor of $S$-vectors, so $\widetilde{g}$ is decomposable.  Height (i) follows from primitivity of the syzygy presentation.

The package method `A.unified_saito_diagnostic(G)` reports both invariants and (with `verify=True`) checks against actual generation.

| $p$ | $k=p-\ell$ | (ii) | what (i)+(ii) reduces to |
|---|---|---|---|
| $\ell$ | $0$ | vacuous (every scalar is decomposable) | $\det M/Q\in\mathbb{K}^*$ — Saito's classical theorem |
| $\ell+1$ | $1$ | vacuous (every $1$-vector is decomposable) | $\operatorname{ht}\ge 3$ — Theorem 1.1 |
| $\ell+2$ | $2$ | non-trivial: rank-$\le 2$ Plücker quadrics on $g_I$ | necessary, **not sufficient** (see counterexample below) |
| $\ge\ell+3$ | $\ge 3$ | higher Plücker relations on $g_I$ | open |

The cells below verify (i)+(ii) on a catalogue of cases and exhibit the counterexample showing necessity is strict only — no purely $g_I$-based sufficient condition is known for $k\ge 2$.


### Uniform $g_I$-only check across $k=0, 1, 2$

For a battery of arrangements we compute (i) the height and (ii) decomposability of the Plücker tensor $\widetilde{g}$, using the package methods `scaled_minor_tensor`, `is_plucker_decomposable`, and the unified diagnostic.  For free arrangements the height is $\infty$ (the principal ideal is the unit ideal); for the rest we expect $\operatorname{ht}=3$ on the minimal generating set.  $\widetilde{g}$ is automatically decomposable for $k\le 1$ and is a non-trivial check for $k=2$.


In [ ]:
# Battery: minimal generating sets across k = 0, 1, 2.
def report_unified(A, label):
    G = list(A.minimal_generators())
    p, ell = len(G), A.n
    k = p - ell
    res = A.unified_saito_diagnostic(generators=G, verify=True, verbose=False)
    flag = ""
    if res.get('mismatch'):
        flag = "  *** mismatch (necessary holds, does not generate) ***"
    print(f"  {label:38s}  ell={ell}  p={p}  k={k}  ht={res['height']:>4}  "
          f"decomposable={res['decomposable']}  generates={res['actually_generates']}{flag}")
    return res

print("=== Battery: minimal generating sets ===\n")
print(f"  {'arrangement':38s}  ell  p   k   ht  decomposable  generates")
print("  " + "-" * 95)

# k = 0 (free arrangements).
report_unified(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1]]),
               "free dim 3 (Boolean)")
report_unified(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,-1,0],[1,0,-1],[0,1,-1]]),
               "free dim 3 (A_2)")
report_unified(HyperplaneArrangement(coordinate_vectors(4)),
               "free dim 4 (Boolean)")

# k = 1 (SPOG / POG).
report_unified(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2]]),
               "SPOG dim 3")
report_unified(HyperplaneArrangement(coordinate_vectors(4)+[[1,1,1,0]]),
               "SPOG dim 4")
report_unified(HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[3,1,-2],[3,1,0],[-4,-4,-3]]),
               "POG-not-SPOG dim 3")

# k = 2 (PD=1 with ell+2 minimal generators).
report_unified(HyperplaneArrangement([[1,-2,0],[2,1,1],[0,1,0],
                                      [2,-1,2],[-1,0,-1],[-2,2,0],[2,2,-1]]),
               "PD=1 dim 3 (Example 10)")

### Inspecting the Plücker matrix and its decomposition for $k=2$

For $k=2$ the antisymmetric matrix $\widetilde{g}$ encodes all the $g_I$.  The cofactor identity $\widetilde{g}\cdot M=0$ holds without assumptions, so the rows of $\widetilde{g}$ are *automatic* syzygies of $G$.  Under (i)+(ii), we can recover an honest $2\times p$ syzygy matrix $N$ as any two $\operatorname{Frac}(S)$-independent rows of $\widetilde{g}$.

The Example-10 arrangement is the prototypical "good" case.


In [ ]:
# Pluecker matrix and decomposition for the Example-10 arrangement.
A_pd1 = HyperplaneArrangement([[1,-2,0],[2,1,1],[0,1,0],[2,-1,2],[-1,0,-1],[-2,2,0],[2,2,-1]])
G_pd1 = list(A_pd1.minimal_generators())
M_pd1 = matrix([A_pd1._to_vector(g) for g in G_pd1])
gtilde = A_pd1.scaled_minor_matrix(G_pd1)

print(f"ell={A_pd1.n}, p={len(G_pd1)}, k={len(G_pd1) - A_pd1.n}")
print(f"\nscaled_minor_matrix is antisymmetric: {gtilde.transpose() == -gtilde}")
print(f"cofactor identity gtilde * M == 0:    {(gtilde * M_pd1).is_zero()}")

rels = A_pd1.plucker_relations(G_pd1)
print(f"\n{len(rels)} Pluecker quadrics, "
      f"{sum(1 for v in rels.values() if v == 0)} vanish "
      f"(decomposable iff all vanish)")
print(f"is_plucker_decomposable: {A_pd1.is_plucker_decomposable(G_pd1)}")

F = A_pd1.S.fraction_field()
print(f"rank of gtilde over Frac(S): {gtilde.change_ring(F).rank()}")

# Recover N as two F-independent rows of gtilde.
nonzero = [a for a in range(gtilde.nrows()) if not gtilde.row(a).is_zero()]
chosen = None
for i, a in enumerate(nonzero):
    for b in nonzero[i+1:]:
        if matrix([gtilde.row(a), gtilde.row(b)]).change_ring(F).rank() == 2:
            chosen = (a, b); break
    if chosen: break
a, b = chosen
N = matrix(A_pd1.S, [gtilde.row(a), gtilde.row(b)])
print(f"\nRecovered N from rows ({a}, {b}) of gtilde, shape {N.nrows()}x{N.ncols()}.")
print(f"  N * M == 0: {(N * M_pd1).is_zero()}")

### Counterexample: necessity is strict for $k\ge 2$

The bare conjunction "(i) $\operatorname{ht}\ge 3$ and (ii) $\widetilde{g}$ decomposable" is **not sufficient** to conclude that $G$ generates $D(\mathcal{A})$.  The construction below exhibits a $5$-tuple $G_{\mathrm{bad}}\subset D(\mathcal{A})$ ($\ell=3,\,p=\ell+2$) for which (i) and (ii) both hold but $G_{\mathrm{bad}}$ does *not* generate.

Take an SPOG arrangement and let $\theta_\star$ be the level generator (the one with linear cofactor $g_\star$).  Replace $\theta_\star$ by *two* multiples $\alpha\theta_\star,\beta\theta_\star$ for two coprime linear forms $\alpha,\beta$:
$$
G_{\mathrm{bad}} \;=\; (G_{\min}\setminus\{\theta_\star\})\,\cup\,\{\alpha\theta_\star,\,\beta\theta_\star\}.
$$
Then $\langle G_{\mathrm{bad}}\rangle = \langle G_{\min}\setminus\{\theta_\star\}\rangle + \langle\alpha,\beta\rangle\!\cdot\!\theta_\star \subsetneq D(\mathcal{A})$ (since $\theta_\star\notin\langle G_{\mathrm{bad}}\rangle$ unless $\langle\alpha,\beta\rangle=S$).  Yet:

* **(i)** $V(I_\mathcal{A}(G_{\mathrm{bad}}))=V(g_\star,\alpha,\beta)\cup V(\text{other }g_I)$, which has codim $3$ — so $\operatorname{ht}\ge 3$ holds.
* **(ii)** $\widetilde{g}$ has rank $2$ over $\operatorname{Frac}(S)$ (and is even decomposable with constant $c\in\mathbb{K}^*$), so all Plücker quadrics vanish.

The trivial syzygy $\beta(\alpha\theta_\star)-\alpha(\beta\theta_\star)=0$ contributes a third minimal syzygy of $G_{\mathrm{bad}}$ that lives outside the rank-$2$ submodule $\operatorname{im}(\text{rows of }\widetilde{g})$, so the rows of $\widetilde{g}$ do **not** $S$-span $\operatorname{syz}(G_{\mathrm{bad}})$.  Equivalently: even when $\widetilde{g}=c\,n_1\wedge n_2$ with $c\in\mathbb{K}^*$, the Buchsbaum–Eisenbud / Hilbert–Burch acyclicity criterion cannot be invoked to upgrade $\langle n_1,n_2\rangle\subseteq\operatorname{syz}(G_{\mathrm{bad}})$ to equality, because the relevant unscaled-minor ideal $I_\ell(M)=Q\cdot I_\mathcal{A}(G)$ always has height at most $1$.


In [ ]:
# Construct G_bad on the SPOG arrangement of Example 2, perturb the level generator.
A = HyperplaneArrangement([[1,0,0],[0,1,0],[0,0,1],[1,1,1],[1,1,2]])
G_min = list(A.minimal_generators())
coeffs = A.saito_coefficients()
star = next(i for i, g in enumerate(coeffs) if g != 0 and g.degree() == 1)
print(f"SPOG cofactor degrees: {[g.degree() if g != 0 else -1 for g in coeffs]}")
print(f"Linear cofactor at index {star}: g_{star} = {coeffs[star]}")

x = A.v
theta_star = G_min[star]
alpha, beta = x[1], x[2]
G_bad = ([g for i, g in enumerate(G_min) if i != star]
         + [alpha * theta_star, beta * theta_star])
print(f"|G_bad| = {len(G_bad)}  (= ell+2 = 5)")
print()

print("--- unified_saito_diagnostic on G_bad ---")
res = A.unified_saito_diagnostic(generators=G_bad, verify=True, verbose=True)

# Show that the minimal syzygies of G_bad have THREE generators, but the rows of
# tilde g span only a rank-2 submodule of syz(G_bad).
M_bad = matrix([A._to_vector(g) for g in G_bad])
mins = syz(Sequence([module_elem(A.S**A.n, tuple(M_bad[i])) for i in range(len(G_bad))]))
print(f"\n# minimal syzygies of G_bad:                            {len(mins)}")
gtilde_bad = A.scaled_minor_matrix(G_bad)
F = A.S.fraction_field()
print(f"rank of scaled_minor_matrix over Frac(S):              "
      f"{gtilde_bad.change_ring(F).rank()}")
print(f"\nThe trivial syzygy beta*(alpha*theta_star) - alpha*(beta*theta_star) = 0")
print(f"contributes a 3rd minimal generator of syz(G_bad) outside the row span")
print(f"of scaled_minor_matrix (over S).  Hence (i)+(ii) hold but G_bad fails to generate.")

### Where this leaves the unified statement

The picture is now clean for $k\le 1$ and partial for $k\ge 2$:

* $k\le 1$: (i) alone is equivalent to "$G$ generates $D(\mathcal{A})$ and the cofactor identity recovers the syzygy module".  This is Saito's classical theorem ($k=0$) and Theorem 1.1 ($k=1$) of the paper.  Condition (ii) is automatic.

* $k\ge 2$: (i) and (ii) are *both* necessary, with (ii) a non-trivial set of Grassmann–Plücker quadrics on the $g_I$.  The catalogue of Example 10 shows examples where (i)+(ii)+generation all hold simultaneously; the counterexample $G_{\mathrm{bad}}$ above shows that (i)+(ii) can hold while generation fails.  The gap reflects the fact that the antisymmetric matrix $\widetilde{g}$ can decompose as $c\cdot n_1\wedge n_2$ with $c\in\mathbb{K}^*$ but $\langle n_1,n_2\rangle\subsetneq\operatorname{syz}(G)$, and Buchsbaum–Eisenbud acyclicity does not close that gap because the relevant unscaled-minor ideal always has height $\le 1$ (it contains $Q$).

A purely $g_I$-only sufficient condition for $k\ge 2$ — equivalently, an intrinsic characterisation of when "$\widetilde{g}$ decomposable with $c\in\mathbb{K}^*$" can be upgraded to "$\langle n_1,\ldots,n_k\rangle = \operatorname{syz}(G)$" — remains open.  One natural candidate is to require that **no row of $\widetilde{g}$ is identically zero** (which fails for $G_{\mathrm{bad}}$: row $0$ is zero), but this has not been tested across enough cases to be conjectural.

**Slogan.**  *"$G$ generates $D(\mathcal{A})$ implies $\operatorname{ht}\,I_\mathcal{A}(G)\ge 3$ and $\widetilde{g}\in\bigwedge^{p-\ell} S^p$ is decomposable; the converse is true for $k\le 1$ (Theorem 1.1) and false in general for $k\ge 2$."*


## Summary of the experiments

* **Extended investigation of Example 10.** For the PD=1, $\ell+2$ regime, the minimal cofactors satisfy a Plücker identity $g_I = c\cdot\varepsilon(I)\cdot \det N_{[\ell+2]\setminus I}$ with $c\in\mathbb{K}^*$, and $\operatorname{ht}\,I_\mathcal{A}(G)=3$ (matching the Example-11 bound at $\operatorname{pd}=1$) on every catalogue entry.

* **Unified statement (Example 12).** Bundle the $g_I$ into the alternating $(p-\ell)$-tensor $\widetilde{g}\in\bigwedge^{p-\ell} S^p$ (`A.scaled_minor_tensor(G)`).  Then $G$ generating $D(\mathcal{A})$ *implies* both $\operatorname{ht}\,I_\mathcal{A}(G)\ge 3$ and decomposability of $\widetilde{g}$ — equivalently, the Grassmann–Plücker quadrics on the $g_I$ vanish (`A.is_plucker_decomposable(G)`).  These two $g_I$-only conditions are a *complete* sufficient condition for $k=p-\ell\le 1$, recovering Saito's classical theorem ($k=0$) and Theorem 1.1 ($k=1$).  For $k\ge 2$ the same conditions are necessary but **not sufficient**: the counterexample $G_{\mathrm{bad}}$ (perturbing the SPOG level generator by two coprime linear multiples) satisfies both yet fails to generate.  The shortfall reflects that decomposability identifies the rows of `A.scaled_minor_matrix(G)` as syzygies but they need not $S$-span $\operatorname{syz}(G)$; Buchsbaum–Eisenbud acyclicity cannot close the gap because $I_\ell(M)=Q\cdot I_\mathcal{A}(G)$ always has height $\le 1$.  A purely $g_I$-only sufficient condition for $k\ge 2$ remains open.
